# Huấn luyện phân đoạn chướng ngại vật trên Kaggle

Notebook này fine-tune YOLO segmentation trên GPU Kaggle. Hiện bộ dữ liệu trong dự án chỉ có `train`; notebook sẽ báo dừng nếu chưa có `val` từ địa điểm/phiên quay riêng. Không chuyển ngẫu nhiên khung hình cùng video sang `val`.

Trước khi chạy: tải dataset đã gán nhãn lên Kaggle, thêm vào **Input**; trong **Settings → Accelerator** chọn GPU và bật **Internet** để cài Ultralytics/tải trọng số pretrained. Kaggle có giới hạn sử dụng GPU theo tài khoản và tình trạng cấp phát.

Video Pexels hiện có trong dự án không được dùng cho ML theo ghi chú quyền sử dụng tại `data/videos/SOURCES.md`. Dùng video tự quay hoặc nguồn có quyền huấn luyện rõ ràng.


In [ ]:
from pathlib import Path
import shutil, zipfile, csv, torch

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA = WORK / "obstacle_dataset"

# Nếu Dataset Kaggle chứa file ZIP, giải nén để tìm session_manifest.csv.
zip_files = list(INPUT_ROOT.rglob("*.zip"))
if zip_files:
    extract_root = WORK / "uploaded_dataset"
    extract_root.mkdir(parents=True, exist_ok=True)
    for z in zip_files:
        with zipfile.ZipFile(z) as archive:
            for member in archive.infolist():
                member_path = Path(member.filename)
                if member_path.is_absolute() or ".." in member_path.parts:
                    raise ValueError(f"Đường dẫn không an toàn trong ZIP: {member.filename}")
            archive.extractall(extract_root)

manifest_files = list(INPUT_ROOT.rglob("session_manifest.csv")) + list((WORK / "uploaded_dataset").rglob("session_manifest.csv"))
if not manifest_files:
    raise FileNotFoundError("Không tìm thấy session_manifest.csv. Hãy thêm dataset đã giải nén hoặc file ZIP vào Input.")
source_root = manifest_files[0].parent
if DATA.exists(): shutil.rmtree(DATA)
shutil.copytree(source_root, DATA)
print("Dataset:", DATA)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "chưa bật")
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Settings → Accelerator rồi chạy lại notebook.")


In [ ]:
# Cài phiên bản Ultralytics tương thích với script trong dự án.
%pip install -q ultralytics==8.4.178


In [ ]:
from collections import defaultdict
from ultralytics import YOLO

CLASS_NAMES = ["person", "bicycle", "motorcycle", "car", "bus_truck", "dog", "pole_tree", "barrier", "walkway", "road", "curb_step", "pothole_drain"]
with (DATA / "session_manifest.csv").open(newline="", encoding="utf-8-sig") as f:
    records = list(csv.DictReader(f))
if not records: raise ValueError("Manifest rỗng.")

# Kiểm tra không trộn cùng phiên quay/địa điểm giữa các tập.
session_split, location_split = {}, {}
counts = defaultdict(int)
positive_counts = defaultdict(int)
for row in records:
    split, session, location = row["split"], row["session_id"], row["location_id"]
    if split not in {"train", "val", "test"}: raise ValueError(f"Split không hợp lệ: {split}")
    if session in session_split and session_split[session] != split: raise ValueError(f"Phiên quay bị chia qua các tập: {session}")
    if location in location_split and location_split[location] != split: raise ValueError(f"Địa điểm bị chia qua các tập: {location}")
    session_split[session] = split; location_split[location] = split
    img = DATA / row["image_path"]
    label = DATA / "labels" / Path(*Path(row["image_path"]).parts[1:]).with_suffix(".txt")
    if not img.is_file() or not label.is_file(): raise FileNotFoundError(f"Thiếu ảnh/nhãn: {img}")
    counts[split] += 1
    for line_no, line in enumerate(label.read_text(encoding="utf-8").splitlines(), 1):
        p = line.split()
        if len(p) < 7 or (len(p)-1) % 2: raise ValueError(f"Polygon lỗi: {label}:{line_no}")
        cid = int(p[0]); xy = list(map(float, p[1:]))
        positive_counts[split] += 1
        if cid not in range(len(CLASS_NAMES)) or any(v < 0 or v > 1 for v in xy): raise ValueError(f"Nhãn/tọa độ không hợp lệ: {label}:{line_no}")

if counts["train"] == 0 or counts["val"] == 0:
    raise RuntimeError("Chưa thể train: cần ảnh có nhãn ở cả train và val. Hãy thêm video val từ địa điểm riêng, không dùng các video Pexels hiện tại.")
if positive_counts["val"] == 0:
    raise RuntimeError("Tập val chưa có đa giác dương tính. Gán nhãn đối tượng trong video val trước khi huấn luyện.")
print("Ảnh theo tập:", dict(counts))

yaml_path = DATA / "dataset.yaml"
yaml_path.write_text("\n".join([f'path: "{DATA.as_posix()}"', "train: images/train", "val: images/val", "test: images/test", "names:"] + [f'  {i}: "{n}"' for i,n in enumerate(CLASS_NAMES)]) + "\n", encoding="utf-8")


In [ ]:
model = YOLO("yolo11n-seg.pt")  # Kaggle Internet cần bật để tải trọng số pretrained lần đầu
run_dir = WORK / "training_runs" / "walkway_obstacles_seg"
if run_dir.exists(): shutil.rmtree(run_dir)
model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=4,
    workers=2,
    device=0,
    project=str(WORK / "training_runs"),
    name="walkway_obstacles_seg",
    exist_ok=True,
    patience=15,
    seed=42,
)

best = WORK / "training_runs" / "walkway_obstacles_seg" / "weights" / "best.pt"
if not best.is_file(): raise FileNotFoundError(f"Không tìm thấy checkpoint tốt nhất: {best}")
print("Model đã huấn luyện:", best)


## Lấy model đã train
Sau khi notebook chạy xong, mở tab **Output** của phiên bản notebook đã lưu để tải `training_runs/walkway_obstacles_seg/weights/best.pt`. Kaggle lưu output trong `/kaggle/working`; hãy dùng **Save Version → Save & Run All** để tạo phiên bản có artifact tải xuống.
